# FraudFinder - Feature Serving FeatureStore

## Overview

This series of labs are updated upon [FraudFinder](https://github.com/googlecloudplatform/fraudfinder) repository which builds a end-to-end real-time fraud detection system on Google Cloud. Throughout the FraudFinder labs, you will learn how to read historical bank transaction data stored in data warehouse, read from a live stream of new transactions, perform exploratory data analysis (EDA), do feature engineering, ingest features into a feature store, train a model using feature store, register your model in a model registry, evaluate your model, deploy your model to an endpoint, do real-time inference on your model with feature store, and monitor your model.


In this notebook, we'll focus on a critical step in any machine learning project: **feature engineering**. You'll learn how to transform raw transaction data into meaningful features that can be used to train a powerful fraud detection model. We'll be using BigQuery for batch feature engineering and Vertex AI Feature Store to manage and serve our features.

### Objective

The primary goal of this notebook is to introduce you to the concepts of batch feature engineering and the Vertex AI Feature Store. You will learn how to:
* **Leverage the Vertex AI Feature Store for efficient feature management.** You'll learn how to create a feature store, define feature groups, and ingest your newly created features for both training and online serving.
* **Prepare for real-time features for inference.** We'll also set the stage for the next notebook in this series, where you'll learn how to create streaming features using Dataflow.

This lab uses the following Google Cloud services and resources:

- [Vertex AI](https://cloud.google.com/vertex-ai/)
- [BigQuery](https://cloud.google.com/bigquery/)
- [Vertex AI Feature Store](https://cloud.google.com/vertex-ai/docs/featurestore/latest/overview)

Steps performed in this notebook:

- Build customer and terminal-related features
- Create Feature store for serve features for inference
- Ingest feature values in Feature store from BigQuery table
- Read features from the feature store

### Load configuration settings from the setup notebook

Set the constants used in this notebook and load the config settings from the `00_environment_setup.ipynb` notebook.

In [1]:
GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
config = !gsutil cat gs://{BUCKET_NAME}/config/notebook_env.py
print(config.n)
exec(config.n)


BUCKET_NAME          = "qwiklabs-asl-02-f5c81170e4c5-fraudfinder"
PROJECT              = "qwiklabs-asl-02-f5c81170e4c5"
REGION               = "us-central1"
ID                   = "vjs6h"
FEATURESTORE_ID      = "fraudfinder_vjs6h"
MODEL_NAME           = "ff_model"
ENDPOINT_NAME        = "ff_model_endpoint"
TRAINING_DS_SIZE     = "1000"



### Import libraries

In [2]:
# General
import datetime as dt
import json
import os
import random
import sys
import time
from datetime import datetime, timedelta
from typing import List, Union

# Data Engineering
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 500)

# Vertex AI and Vertex AI Feature Store
from google.cloud import aiplatform as vertex_ai
from google.cloud import bigquery

/home/jupyter/new_ff/asl-ml-immersion/asl_mlops/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


### Define constants

In [3]:
# Define the date range of transactions for feature engineering (last 10 days up until yesterday)
YESTERDAY = datetime.today() - timedelta(days=1)
YEAR_MONTH_PREFIX = YESTERDAY.strftime("%Y-%m")
DATAPROCESSING_START_DATE = (YESTERDAY - timedelta(days=10)).strftime(
    "%Y-%m-%d"
)
DATAPROCESSING_END_DATE = YESTERDAY.strftime("%Y-%m-%d")

# Define BiqQuery dataset and tables to calculate features.
RAW_BQ_TRANSACTION_TABLE_URI = f"{PROJECT_ID}.tx.tx"

INGESTION_BQ_TRANSACTION_TABLE_URI = f"{PROJECT_ID}.tx.ingestion_tx_records"
INGESTION_BQ_LABELS_TABLE_URI = f"{PROJECT_ID}.tx.ingestion_tx_labels"

RAW_BQ_LABELS_TABLE_URI = f"{PROJECT_ID}.tx.txlabels"
FEATURES_BQ_TABLE_URI = f"{PROJECT_ID}.tx.wide_features_table"

# Define Vertex AI Feature store tables and views.

CUSTOMERS_FE_BQ_VIEW_URI = f"{PROJECT_ID}.tx.v_customers_features"
CUSTOMERS_FE_BQ_BATCH_TABLE_URI = f"{PROJECT_ID}.tx.t_customers_batch_features"

TERMINALS_TABLE_NAME = f"terminals_{DATAPROCESSING_END_DATE.replace('-', '')}"

TERMINALS_FE_BQ_VIEW_URI = f"{PROJECT_ID}.tx.v_terminals_features"
TERMINALS_FE_BQ_BATCH_TABLE_URI = f"{PROJECT_ID}.tx.t_terminals_batch_features"

CUSTOMERS_STREAMING_FE_TABLE_URI = (
    f"{PROJECT_ID}.tx.t_customers_streaming_features"
)
TERMINALS_STREAMING_FE_TABLE_URI = (
    f"{PROJECT_ID}.tx.t_terminals_streaming_features"
)

ONLINE_STORAGE_NODES = 1
FEATURE_TIME = "feature_ts"
CUSTOMER_ENTITY_ID = "customer"
TERMINAL_ENTITY_ID = "terminal"

### Initialize Vertex AI SDK

Initialize the Vertex AI SDK to get access to Vertex AI services programmatically. 

In [4]:
vertex_ai.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_NAME)

## Managing Features with Vertex AI Feature Store

Now that we've engineered our features, we need a robust and efficient way to manage them. This is where the **Vertex AI Feature Store** comes in. A feature store is a centralized repository for storing, serving, and managing machine learning features. It plays a crucial role in the MLOps lifecycle by providing a single source of truth for features, which helps to ensure consistency between training and serving, prevent feature leakage, and promote feature reuse across different models and projects.

### Key Concepts in Vertex AI Feature Store

Before we start creating our feature store, let's quickly go over some key concepts:

*   **Feature Store**: A top-level container for organizing and managing your features.
*   **Entity Type**: A collection of semantically related features. In our case, we'll have two entity types: `customer` and `terminal`.
*   **Feature**: A measurable property or characteristic of an entity. For example, `customer_id_nb_tx_1day_window` is a feature of the `customer` entity type.
*   **Feature View**: A logical view of features from a data source. It defines how features are synced from the data source to the online store for serving.

### Benefits of Using a Feature Store

Using a feature store offers several advantages, including:

*   **Preventing Training-Serving Skew**: By using the same feature store for both training and serving, you can ensure that your model is using the exact same features in both environments, which helps to prevent performance degradation due to inconsistencies.
*   **Promoting Feature Reuse**: A centralized feature store makes it easy to discover and reuse existing features across different models and teams, which can save time and effort.
*   **Improving Model Governance**: A feature store provides a centralized place to track feature lineage and metadata, which can help with model explainability and compliance.

In the following cells, we'll create a feature store, define our entity types and features, and ingest our batch features from BigQuery.

### Import libraries

In [5]:
from google.cloud import bigquery
from google.cloud.aiplatform_v1 import (
    FeatureOnlineStoreAdminServiceClient,
    FeatureOnlineStoreServiceClient,
    FeatureRegistryServiceClient,
)
from google.cloud.aiplatform_v1.types import feature as feature_pb2
from google.cloud.aiplatform_v1.types import feature_group as feature_group_pb2
from google.cloud.aiplatform_v1.types import (
    feature_online_store as feature_online_store_pb2,
)
from google.cloud.aiplatform_v1.types import (
    feature_online_store_admin_service as feature_online_store_admin_service_pb2,
)
from google.cloud.aiplatform_v1.types import (
    feature_online_store_service as feature_online_store_service_pb2,
)
from google.cloud.aiplatform_v1.types import (
    feature_registry_service as feature_registry_service_pb2,
)
from google.cloud.aiplatform_v1.types import feature_view as feature_view_pb2
from google.cloud.aiplatform_v1.types import (
    featurestore_service as featurestore_service_pb2,
)
from google.cloud.aiplatform_v1.types import io as io_pb2

### Initialize Admin Service Client

In [6]:
API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"

In [7]:
admin_client = FeatureOnlineStoreAdminServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)
registry_client = FeatureRegistryServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)

### Create online store instance

To create an online store instance.
Create a `FeatureOnlineStore` instance with autoscaling.

In [8]:
online_store_config = feature_online_store_pb2.FeatureOnlineStore(
    bigtable=feature_online_store_pb2.FeatureOnlineStore.Bigtable(
        auto_scaling=feature_online_store_pb2.FeatureOnlineStore.Bigtable.AutoScaling(
            min_node_count=1, max_node_count=1, cpu_utilization_target=50
        )
    )
)

create_store_lro = admin_client.create_feature_online_store(
    feature_online_store_admin_service_pb2.CreateFeatureOnlineStoreRequest(
        parent=f"projects/{PROJECT_ID}/locations/{REGION}",
        feature_online_store_id=FEATURESTORE_ID,
        feature_online_store=online_store_config,
    )
)

### Verify online store instance creation

After the long-running operation (LRO) is complete, show the result.

> **Note:** This operation might take up to 10 minutes to complete.

In [9]:
# Wait for the LRO to finish and get the LRO result.
print(create_store_lro.result())

name: "projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_vjs6h"



In [10]:
# Use list to verify the store is created.
admin_client.list_feature_online_stores(
    parent=f"projects/{PROJECT_ID}/locations/{REGION}"
)

ListFeatureOnlineStoresPager<feature_online_stores {
  name: "projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_atwc3"
  create_time {
    seconds: 1777310446
    nanos: 222128000
  }
  update_time {
    seconds: 1777310446
    nanos: 311769000
  }
  etag: "AMEw9yOFVu17a2oeH5ctS6tiNzSNE25nQXe_dkK748Jzz2dOuV1ME2V9gZrlOsTxFzHO"
  bigtable {
    auto_scaling {
      min_node_count: 1
      max_node_count: 1
      cpu_utilization_target: 50
    }
    zone: "us-central1-c"
  }
}
feature_online_stores {
  name: "projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_vjs6h"
  create_time {
    seconds: 1777370739
    nanos: 906852000
  }
  update_time {
    seconds: 1777370740
    nanos: 8586000
  }
  etag: "AMEw9yPGvmv7Hi6r6oWeUZoHIoAyOacZNXHS_iogyUB8Bf-evwYHMsf7IoPjywoAEm4="
  bigtable {
    auto_scaling {
      min_node_count: 1
      max_node_count: 1
      cpu_utilization_target: 50
    }
    zone: "us-central1-c"
  }
}
>

### Registering Feature Groups and Features

Before we can use our features for online serving, we need to register their metadata with the Vertex AI Feature Registry. This involves two main concepts:

*   **`FeatureGroup`**: A `FeatureGroup` is a logical container that groups features defined on the same BigQuery data source. It tells the Feature Store where your feature data is located (the `input_uri`) and which column contains the unique entity IDs.

*   **`Feature`**: A `Feature` represents a single column (a feature) within a `FeatureGroup`'s data source.

In our project, we have four distinct sets of features based on the entity type (customer or terminal) and the calculation method (batch or streaming). Therefore, we will create four `FeatureGroup`s to organize them, one for each of our BigQuery feature tables:
1.  **`fraudfinder_customers_batch`**: For batch-calculated customer features.
2.  **`fraudfinder_customers_streaming`**: For streaming-calculated customer features (currently empty).
3.  **`fraudfinder_terminals_batch`**: For batch-calculated terminal features.
4.  **`fraudfinder_terminals_streaming`**: For streaming-calculated terminal features (currently empty).

The following cells will use our helper function to create these four `FeatureGroup`s and register all the associated `Feature`s (columns) within each one.

#### Define utility method for feature groups creation

In [11]:
def create_fs_feature_group(
    bq_source_uri, entity_id_column, feature_group_id, feature_ids_list
):

    # Now, create the featureGroup
    feature_group_config = feature_group_pb2.FeatureGroup(
        big_query=feature_group_pb2.FeatureGroup.BigQuery(
            big_query_source=io_pb2.BigQuerySource(
                input_uri=f"bq://{bq_source_uri}"
            ),
            # Add the entity_id_columns parameter here
            entity_id_columns=[entity_id_column],
        )
    )
    create_group_lro = registry_client.create_feature_group(
        feature_registry_service_pb2.CreateFeatureGroupRequest(
            parent=f"projects/{PROJECT_ID}/locations/{REGION}",
            feature_group_id=feature_group_id,
            feature_group=feature_group_config,
        )
    )

    # After the long-running operation (LRO) is complete, show the result.
    print(create_group_lro.result())

    create_feature_lros = []
    for id in feature_ids_list:
        create_feature_lros.append(
            registry_client.create_feature(
                featurestore_service_pb2.CreateFeatureRequest(
                    parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureGroups/{feature_group_id}",
                    feature_id=id,
                    feature=feature_pb2.Feature(),
                )
            )
        )

    # Wait for FS Group creation
    for lro in create_feature_lros:
        # After the long-running operation (LRO) is complete, show the result.
        print(lro.result())

In [12]:
CUSTOMER_ID_COLUMN = "entity_id"  # entity_id

CUSTOMER_BATCH_FEATURES_GROUP_ID = "fraudfinder_customers_batch"

CUSTOMER_BATCH_FEATURE_IDS = [
    "customer_id_nb_tx_14day_window",
    "customer_id_avg_amount_7day_window",
    "customer_id_nb_tx_1day_window",
    "customer_id_avg_amount_1day_window",
    "customer_id_avg_amount_14day_window",
    "customer_id_nb_tx_7day_window",
]

# Creating feature Group for batch for customers
create_fs_feature_group(
    bq_source_uri=CUSTOMERS_FE_BQ_BATCH_TABLE_URI,
    entity_id_column=CUSTOMER_ID_COLUMN,
    feature_group_id=CUSTOMER_BATCH_FEATURES_GROUP_ID,
    feature_ids_list=CUSTOMER_BATCH_FEATURE_IDS,
)

AlreadyExists: 409 FeatureGroup `fraudfinder_customers_batch` already exists in projects/qwiklabs-asl-02-f5c81170e4c5/locations/us-central1.

In [ ]:
CUSTOMER_STREAMING_FEATURES_GROUP_ID = "fraudfinder_customers_streaming"
CUSTOMER_STREAMING_FEATURE_IDS = [
    "customer_id_nb_tx_15min_window",
    "customer_id_nb_tx_30min_window",
    "customer_id_nb_tx_60min_window",
    "customer_id_avg_amount_15min_window",
    "customer_id_avg_amount_30min_window",
    "customer_id_avg_amount_60min_window",
]

# Creating feature Group for streaming for customers
create_fs_feature_group(
    bq_source_uri=CUSTOMERS_STREAMING_FE_TABLE_URI,
    entity_id_column=CUSTOMER_ID_COLUMN,
    feature_group_id=CUSTOMER_STREAMING_FEATURES_GROUP_ID,
    feature_ids_list=CUSTOMER_STREAMING_FEATURE_IDS,
)

In [ ]:
# Now, create the featureGroup for terminals
TERMINAL_ID_COLUMN = "entity_id"

TERMINAL_BATCH_FEATURES_GROUP_ID = "fraudfinder_terminals_batch"
TERMINAL_BATCH_FEATURE_IDS = [
    "terminal_id_nb_tx_1day_window",
    "terminal_id_nb_tx_7day_window",
    "terminal_id_nb_tx_14day_window",
    "terminal_id_risk_1day_window",
    "terminal_id_risk_7day_window",
    "terminal_id_risk_14day_window",
]

# Creating feature Group for batch for customers
create_fs_feature_group(
    bq_source_uri=TERMINALS_FE_BQ_BATCH_TABLE_URI,
    entity_id_column=TERMINAL_ID_COLUMN,
    feature_group_id=TERMINAL_BATCH_FEATURES_GROUP_ID,
    feature_ids_list=TERMINAL_BATCH_FEATURE_IDS,
)

In [ ]:
# Now, create the featureGroup for terminals streaming features
TERMINAL_STREAMING_FEATURES_GROUP_ID = "fraudfinder_terminals_streaming"
TERMINAL_STREAMING_FEATURE_IDS = [
    "terminal_id_nb_tx_15min_window",
    "terminal_id_nb_tx_30min_window",
    "terminal_id_nb_tx_60min_window",
    "terminal_id_avg_amount_15min_window",
    "terminal_id_avg_amount_30min_window",
    "terminal_id_avg_amount_60min_window",
]

# Creating feature Group for batch for customers
create_fs_feature_group(
    bq_source_uri=TERMINALS_STREAMING_FE_TABLE_URI,
    entity_id_column=TERMINAL_ID_COLUMN,
    feature_group_id=TERMINAL_STREAMING_FEATURES_GROUP_ID,
    feature_ids_list=TERMINAL_STREAMING_FEATURE_IDS,
)

### Connecting Feature Groups to the Online Store with `FeatureView`

Now that we have registered our `FeatureGroup`s (our feature sources), we need a way to tell the Feature Store to actually serve these features for real-time lookups. This is the job of a **`FeatureView`**.

A `FeatureView` acts as a bridge between the feature sources (`FeatureGroup`s) and the online serving cluster (the `FeatureOnlineStore` we created earlier). It defines which features from which feature groups should be made available for low-latency retrieval.

Key responsibilities of a `FeatureView`:
*   **Linking Sources:** It links one or more `FeatureGroup`s to a specific `FeatureOnlineStore`.
*   **Syncing Data:** It manages the synchronization of data from the BigQuery sources (defined in the `FeatureGroup`s) to the high-performance online store (Bigtable). You can configure this sync to run on a schedule (cron) or continuously. For this lab, we'll use a continuous sync to keep the online data as fresh as possible.

We will create two `FeatureView`s, one for each entity type:
1.  **`fv_fraudfinder_customers`**: This view will combine the batch and streaming features for customers.
2.  **`fv_fraudfinder_terminals`**: This view will combine the batch and streaming features for terminals.

The helper function below will create these views and start the data sync process.

In [ ]:
def create_online_fs_view(
    fs_view_id,
    fs_online_store_id,
    feature_group_ids,
    feature_ids_list,
    continuous,
    cron_schedule=None,
):

    feature_groups = []

    for feature_group_id, feature_ids in zip(
        feature_group_ids, feature_ids_list
    ):
        feature_groups.append(
            feature_view_pb2.FeatureView.FeatureRegistrySource.FeatureGroup(
                feature_group_id=feature_group_id,
                feature_ids=feature_ids,
            )
        )

    feature_registry_source = (
        feature_view_pb2.FeatureView.FeatureRegistrySource(
            feature_groups=feature_groups
        )
    )

    if continuous:
        sync_config = feature_view_pb2.FeatureView.SyncConfig(continuous=True)
    else:
        sync_config = feature_view_pb2.FeatureView.SyncConfig(
            cron=cron_schedule
        )

    create_view_lro = admin_client.create_feature_view(
        feature_online_store_admin_service_pb2.CreateFeatureViewRequest(
            parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{fs_online_store_id}",
            feature_view_id=fs_view_id,
            run_sync_immediately=True,
            feature_view=feature_view_pb2.FeatureView(
                feature_registry_source=feature_registry_source,
                sync_config=sync_config,
            ),
        )
    )

    # Wait for LRO to complete and show result
    print(create_view_lro.result())

#### Test Check to ensure all resources provisioned:

In [17]:
# Test cell to ensure that all resources provisioned:
import time

TEST_C_FEATURE_VIEW_ID = f"fv_customers_test_provisioned"
TEST_T_FEATURE_VIEW_ID = f"fv_terminal_test_provisioned"

test_c_feature_view = f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}/featureViews/{TEST_C_FEATURE_VIEW_ID}"
test_t_feature_view = f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}/featureViews/{TEST_T_FEATURE_VIEW_ID}"

admin_client.list_feature_views(
    parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}"
)

create_online_fs_view(
    fs_view_id=TEST_C_FEATURE_VIEW_ID,
    fs_online_store_id=FEATURESTORE_ID,
    feature_group_ids=[
        CUSTOMER_BATCH_FEATURES_GROUP_ID,
        CUSTOMER_STREAMING_FEATURES_GROUP_ID,
    ],
    feature_ids_list=[
        CUSTOMER_BATCH_FEATURE_IDS,
        CUSTOMER_STREAMING_FEATURE_IDS,
    ],
    continuous=True,
)

create_online_fs_view(
    fs_view_id=TEST_T_FEATURE_VIEW_ID,
    fs_online_store_id=FEATURESTORE_ID,
    feature_group_ids=[
        TERMINAL_BATCH_FEATURES_GROUP_ID,
        TERMINAL_STREAMING_FEATURES_GROUP_ID,
    ],
    feature_ids_list=[
        TERMINAL_BATCH_FEATURE_IDS,
        TERMINAL_STREAMING_FEATURE_IDS,
    ],
    continuous=True,
)

# Delete TEST FeatureView
delete_view_c_lro = admin_client.delete_feature_view(name=test_c_feature_view)
delete_view_t_lro = admin_client.delete_feature_view(name=test_t_feature_view)

print(delete_view_c_lro.result())
print(delete_view_t_lro.result())

admin_client.list_feature_views(
    parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}"
)

name: "projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_customers_test_provisioned"

name: "projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_terminal_test_provisioned"



InternalServerError: 500 Internal error encountered.

#### Creating featurestore view for Customers Features

In [18]:
FS_VIEW_SUFFIX_ID = ""  # Default without suffix
CUSTOMER_FEATURE_VIEW_ID = f"fv_fraudfinder_customers{FS_VIEW_SUFFIX_ID}"

create_online_fs_view(
    fs_view_id=CUSTOMER_FEATURE_VIEW_ID,
    fs_online_store_id=FEATURESTORE_ID,
    feature_group_ids=[
        CUSTOMER_BATCH_FEATURES_GROUP_ID,
        CUSTOMER_STREAMING_FEATURES_GROUP_ID,
    ],
    feature_ids_list=[
        CUSTOMER_BATCH_FEATURE_IDS,
        CUSTOMER_STREAMING_FEATURE_IDS,
    ],
    continuous=True,
)

AlreadyExists: 409 FeatureView `projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_fraudfinder_customers` already exists.

#### Creating featurestore view for Terminals Features:

In [19]:
TERMINAL_FEATURE_VIEW_ID = f"fv_fraudfinder_terminals{FS_VIEW_SUFFIX_ID}"

create_online_fs_view(
    fs_view_id=TERMINAL_FEATURE_VIEW_ID,
    fs_online_store_id=FEATURESTORE_ID,
    feature_group_ids=[
        TERMINAL_BATCH_FEATURES_GROUP_ID,
        TERMINAL_STREAMING_FEATURES_GROUP_ID,
    ],
    feature_ids_list=[
        TERMINAL_BATCH_FEATURE_IDS,
        TERMINAL_STREAMING_FEATURE_IDS,
    ],
    continuous=True,
)

AlreadyExists: 409 FeatureView `projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_fraudfinder_terminals` already exists.

Verify that the `FeatureView` instance is created by listing all the feature views within the online store.

In [20]:
# Again, list all feature view under the FEATURESTORE_ID to confirm
admin_client.list_feature_views(
    parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}"
)

ListFeatureViewsPager<feature_views {
  name: "projects/215829578942/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_fraudfinder_customers"
  create_time {
    seconds: 1777310478
    nanos: 945968000
  }
  update_time {
    seconds: 1777310480
    nanos: 863358000
  }
  etag: "AMEw9yMkVVj9XlDquGFbhX_kASKGQExDJ3zhcUWOA7F4oIxi5xNLZSsvEcNx58TKq7od"
  sync_config {
    continuous: true
  }
  feature_registry_source {
    feature_groups {
      feature_group_id: "fraudfinder_customers_batch"
      feature_ids: "customer_id_nb_tx_14day_window"
      feature_ids: "customer_id_avg_amount_7day_window"
      feature_ids: "customer_id_nb_tx_1day_window"
      feature_ids: "customer_id_avg_amount_1day_window"
      feature_ids: "customer_id_avg_amount_14day_window"
      feature_ids: "customer_id_nb_tx_7day_window"
    }
    feature_groups {
      feature_group_id: "fraudfinder_customers_streaming"
      feature_ids: "customer_id_nb_tx_15min_window"
      feature_ids: 

In [21]:
admin_client.list_feature_view_syncs(
    parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}/featureViews/{CUSTOMER_FEATURE_VIEW_ID}"
)

ListFeatureViewSyncsPager<feature_view_syncs {
  name: "projects/qwiklabs-asl-02-f5c81170e4c5/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_fraudfinder_customers/featureViewSyncs/105207906122596352"
  create_time {
    seconds: 1777310841
    nanos: 377385000
  }
  run_time {
    start_time {
      seconds: 1777310841
      nanos: 377385000
    }
  }
}
feature_view_syncs {
  name: "projects/qwiklabs-asl-02-f5c81170e4c5/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_fraudfinder_customers/featureViewSyncs/5812183427675324416"
  create_time {
    seconds: 1777310480
    nanos: 858656000
  }
  final_status {
    code: 9
    message: "Vertex AI Service Agent service-215829578942@gcp-sa-aiplatform.iam.gserviceaccount.com, does not have required IAM permission(s) to the provided resource BigQuery job ee2e4baaa053e5308-tp:us-central1.aiplatform-5812183427675324416-230947167965: Access Denied: BigQuery BigQuery: Permission denied while 

In [22]:
admin_client.list_feature_view_syncs(
    parent=f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}/featureViews/{TERMINAL_FEATURE_VIEW_ID}"
)

ListFeatureViewSyncsPager<feature_view_syncs {
  name: "projects/qwiklabs-asl-02-f5c81170e4c5/locations/us-central1/featureOnlineStores/fraudfinder_atwc3/featureViews/fv_fraudfinder_terminals/featureViewSyncs/4727295304548745216"
  create_time {
    seconds: 1777310507
    nanos: 208377000
  }
  run_time {
    start_time {
      seconds: 1777310507
      nanos: 208377000
    }
  }
}
>

### Online Serving for Real-Time Fraud Detection

Now that we've ingested our features into the Vertex AI Feature Store, it's time to talk about **online serving**. In the context of fraud detection, online serving refers to the process of retrieving feature values for a single entity (e.g., a customer or a terminal) in real-time, with very low latency. This is a critical requirement for our use case, as we need to be able to make a fraud prediction in a matter of milliseconds, while a customer is waiting for their transaction to be approved.

Vertex AI Feature Store provides a highly scalable and low-latency online serving solution that is optimized for real-time use cases. When you ingest features into a feature store, they are stored in both an offline storage (BigQuery) for batch use cases and an online store (Bigtable) for real-time serving. This dual-storage architecture allows you to use the same features for both training and inference, without having to worry about data skew.

In the following cells, we'll show you how to use the Vertex AI SDK to fetch feature values from the online store.

In [23]:
data_client = FeatureOnlineStoreServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)

The `FeatureView` already defines the features needed for the model (via the BigQuery view in this demo). To fetch the data, submit a `fetch_feature_values` request specifying the `FeatureView` resource path and the ID of the entity.

#### Lets get a customer record from BigQuery View:

In [24]:
%%bigquery customer_view_record --project {PROJECT_ID}
SELECT * 
FROM tx.v_customers_features
ORDER BY feature_timestamp DESC
LIMIT 1

Query is running:   0%|          |

Downloading:   0%|          |

In [25]:
customer_view_record

,feature_timestamp,entity_id,customer_id_nb_tx_1day_window,customer_id_nb_tx_7day_window,customer_id_nb_tx_14day_window,customer_id_avg_amount_1day_window,customer_id_avg_amount_7day_window,customer_id_avg_amount_14day_window
0,2026-04-27 17:54:43.878676+00:00,0001909647407798,1,1,1,65.21,65.21,65.21


#### Now we can check that this record availible using Vertex AI Feature Store online serving
Note: it can take up to 

In [26]:
print(f"Featurestore ID: {FEATURESTORE_ID}")
print(f"Featurestore View ID: {CUSTOMER_FEATURE_VIEW_ID}")

customer_key = customer_view_record["entity_id"][0]
print(f"entity_id={customer_key}")

FEATURE_VIEW_FULL_ID = f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}/featureViews/{CUSTOMER_FEATURE_VIEW_ID}"

try:
    fe_data = data_client.fetch_feature_values(
        request=feature_online_store_service_pb2.FetchFeatureValuesRequest(
            feature_view=FEATURE_VIEW_FULL_ID,
            data_key=feature_online_store_service_pb2.FeatureViewDataKey(
                key=customer_key
            ),
            data_format=feature_online_store_service_pb2.FeatureViewDataFormat.PROTO_STRUCT,
        )
    )
    customer_features = json.dumps(
        {k: v for k, v in fe_data.proto_struct.items()}, indent=4
    )
    print(f"Customer Features: {customer_features}")
except Exception as exp:
    print("ERROR: " + str(exp))

Featurestore ID: fraudfinder_atwc3
Featurestore View ID: fv_fraudfinder_customers
entity_id=0001909647407798
Customer Features: {
    "customer_id_avg_amount_14day_window": 59.71842105263158,
    "customer_id_nb_tx_14day_window": 19.0,
    "customer_id_nb_tx_7day_window": 12.0,
    "customer_id_nb_tx_1day_window": 1.0,
    "customer_id_avg_amount_7day_window": 59.6875,
    "customer_id_avg_amount_1day_window": 64.47
}


In [27]:
%%bigquery terminal_view_record --project {PROJECT_ID}
SELECT * 
FROM tx.v_terminals_features
ORDER BY feature_timestamp DESC
LIMIT 1

Query is running:   0%|          |

Downloading:   0%|          |

In [28]:
terminal_view_record

,feature_timestamp,entity_id,terminal_id_nb_tx_1day_window,terminal_id_nb_tx_7day_window,terminal_id_nb_tx_14day_window,terminal_id_risk_1day_window,terminal_id_risk_7day_window,terminal_id_risk_14day_window
0,2026-04-27 17:54:52.346545+00:00,00064542,0,0,0,0.0,0.0,0.0


In [29]:
print(f"Featurestore ID: {FEATURESTORE_ID}")
print(f"Featurestore View ID: {TERMINAL_FEATURE_VIEW_ID}")

terminal_key = terminal_view_record["entity_id"][0]
print(f"entity_id={terminal_key}")

FEATURE_VIEW_FULL_ID = f"projects/{PROJECT_ID}/locations/{REGION}/featureOnlineStores/{FEATURESTORE_ID}/featureViews/{TERMINAL_FEATURE_VIEW_ID}"

try:
    fe_data = data_client.fetch_feature_values(
        request=feature_online_store_service_pb2.FetchFeatureValuesRequest(
            feature_view=FEATURE_VIEW_FULL_ID,
            data_key=feature_online_store_service_pb2.FeatureViewDataKey(
                key=terminal_key
            ),
            data_format=feature_online_store_service_pb2.FeatureViewDataFormat.PROTO_STRUCT,
        )
    )
    terminal_features = json.dumps(
        {k: v for k, v in fe_data.proto_struct.items()}, indent=4
    )
    print(f"Terminal features: {terminal_features}")
except Exception as exp:
    print("ERROR: " + str(exp))

Featurestore ID: fraudfinder_atwc3
Featurestore View ID: fv_fraudfinder_terminals
entity_id=00064542
Terminal features: {
    "terminal_id_nb_tx_14day_window": 137.0,
    "terminal_id_nb_tx_1day_window": 13.0,
    "terminal_id_risk_7day_window": 0.0,
    "terminal_id_risk_14day_window": 0.0,
    "terminal_id_nb_tx_7day_window": 125.0,
    "terminal_id_risk_1day_window": 0.0
}


### Inspect your feature store in the Vertex AI console

You can also inspect your feature store in the [Vertex AI Feature Store console](https://console.cloud.google.com/vertex-ai/feature-store/online-stores)

### END

Now you can go to the next notebook `04_feature_engineering_streaming.ipynb`

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.